
---

### 1. 完整 ABAP 代码 (Complete ABAP Code)

```abap
  METHOD write_player_details.

    " --- 定义用于追踪分组的哈希表 ---
    TYPES: tt_processed_keys TYPE HASHED TABLE OF string WITH UNIQUE KEY table_line.
    DATA: lt_processed_keys TYPE tt_processed_keys.

    " --- 定义辅助颜色结构 ---
    DATA: lt_s_color TYPE lvc_t_scol,
          ls_s_color TYPE lvc_s_scol.

    " =====================================================================
    " 1. 【核心防御】前台条件强行清洗底层脏数据 (杜绝跨币种污染)
    " =====================================================================
    IF s_waers[] IS NOT INITIAL.
      DELETE me->lt_find WHERE rwcur NOT IN s_waers.
    ENDIF.
    IF s_racct[] IS NOT INITIAL.
      DELETE me->lt_find WHERE racct NOT IN s_racct.
    ENDIF.

    TYPES: BEGIN OF ty_beg_bal,
             rbukrs TYPE acdoca-rbukrs,
             lifnr  TYPE acdoca-lifnr,
             racct  TYPE acdoca-racct,
             rwcur  TYPE acdoca-rwcur,
             rhcur  TYPE acdoca-rhcur,
             beghsl TYPE acdoca-hsl,
             begwsl TYPE acdoca-wsl,
           END OF ty_beg_bal.

    DATA: lt_beg_bal       TYPE TABLE OF ty_beg_bal,
          lt_final_display TYPE TABLE OF ts_s_alvplay,
          ls_sum_month     TYPE ts_s_alvplay,
          ls_sum_year      TYPE ts_s_alvplay.

    DATA: lv_delpoplo TYPE poper,
          lv_delpoper TYPE poper.

    READ TABLE s_monat INTO DATA(ls_monat) INDEX 1.
    IF sy-subrc = 0.
      lv_delpoplo = ls_monat-low.
      lv_delpoper = ls_monat-high.
      IF lv_delpoper IS INITIAL.
        lv_delpoper = ls_monat-low.
      ENDIF.
      IF lv_delpoper > '12'.
        lv_delpoper = '12'.
      ENDIF.
    ENDIF.

    " =====================================================================
    " 2. 极速捞取“期初余额”
    " =====================================================================
    SELECT rbukrs, lifnr, kunnr, racct, rwcur, rhcur,
           SUM( hsl ) AS beghsl,
           SUM( wsl ) AS begwsl
      FROM acdoca
     WHERE rbukrs IN @s_bukrs
       AND racct  IN @s_racct
       AND lifnr  IN @s_lifnr
       AND kunnr  IN @s_kunnr
       AND rwcur  IN @s_waers
       AND ( gjahr < @p_gjahr OR ( gjahr = @p_gjahr AND poper < @lv_delpoplo ) )
     GROUP BY rbukrs, lifnr, kunnr, racct, rwcur, rhcur
      INTO TABLE @DATA(lt_beg_bal_raw).

    LOOP AT lt_beg_bal_raw INTO DATA(ls_raw).
      DATA(ls_beg) = VALUE ty_beg_bal( rbukrs = ls_raw-rbukrs
                                       racct  = ls_raw-racct
                                       rwcur  = ls_raw-rwcur
                                       rhcur  = ls_raw-rhcur
                                       beghsl = ls_raw-beghsl
                                       begwsl = ls_raw-begwsl ).
      IF ls_raw-lifnr IS NOT INITIAL.
        ls_beg-lifnr = ls_raw-lifnr.
      ELSE.
        ls_beg-lifnr = ls_raw-kunnr.
      ENDIF.
      COLLECT ls_beg INTO lt_beg_bal.
    ENDLOOP.

    SORT lt_beg_bal BY rbukrs lifnr racct rwcur rhcur.

    " =====================================================================
    " 3. 准备各类基础数据
    " =====================================================================
    SELECT FROM zfit_ofn AS a
      FIELDS a~bukrs, a~hfc_doc, a~belnr, a~gjahr, a~monat
      WHERE a~bukrs IN @s_bukrs
        AND a~gjahr = @p_gjahr
      INTO TABLE @DATA(lt_zfit_ofn).
    IF sy-subrc = 0. SORT lt_zfit_ofn BY bukrs gjahr belnr. ENDIF.

    SELECT FROM lfa1 AS a
      FIELDS a~lifnr, concat( a~name1, concat( a~name2, a~name3 ) ) AS name
      ORDER BY lifnr INTO TABLE @DATA(lt_lfa1).

    SELECT FROM kna1 AS a
      FIELDS a~kunnr, concat( a~name1, a~name2 ) AS name
      ORDER BY kunnr INTO TABLE @DATA(lt_kna1).

    SELECT * FROM zc_hkont_detail_tf( iv_ktopl = '1000', iv_saknr = '0000000000', iv_spras = '1' )
      INTO TABLE @DATA(lt_skat).
    SORT lt_skat BY ktopl saknr.

    IF me->lt_findk IS NOT INITIAL. APPEND LINES OF me->lt_findk TO me->lt_find. CLEAR: me->lt_findk. ENDIF.
    IF me->lt_findbug IS NOT INITIAL. APPEND LINES OF me->lt_findbug TO me->lt_find. CLEAR: me->lt_findbug. ENDIF.

    DELETE me->lt_find WHERE lifnr = ''.
    SORT me->lt_find BY rldnr rbukrs gjahr belnr docln racct lifnr.
    DELETE ADJACENT DUPLICATES FROM me->lt_find COMPARING rldnr rbukrs gjahr belnr docln racct lifnr.

    DATA: lv_zhuanh TYPE p DECIMALS 3.
    LOOP AT me->lt_find ASSIGNING FIELD-SYMBOL(<ls_find>).
      CLEAR: lv_zhuanh.
      CALL FUNCTION 'CURRENCY_CONVERTING_FACTOR'
        EXPORTING
          currency          = <ls_find>-rwcur
        IMPORTING
          factor            = lv_zhuanh
        EXCEPTIONS
          too_many_decimals = 1
          OTHERS            = 2.
      IF sy-subrc = 0. <ls_find>-wsl = <ls_find>-wsl * lv_zhuanh. ENDIF.
      <ls_find>-rldnr = ''. <ls_find>-docln = ''.

      IF <ls_find>-lifnr IS NOT INITIAL.
        FIND FIRST OCCURRENCE OF REGEX '[一-龥]' IN <ls_find>-lifnr.
        IF sy-subrc = 0. <ls_find>-delf = abap_true. ENDIF.
      ENDIF.
    ENDLOOP.

    DELETE me->lt_find WHERE delf = abap_true.
    DELETE me->lt_find WHERE lifnr = ''.
    DELETE me->lt_find WHERE gjahr <> p_gjahr.
    DELETE me->lt_find WHERE poper > lv_delpoper.
    DELETE me->lt_find WHERE poper < lv_delpoplo.

    SELECT FROM @me->lt_find AS a
      INNER JOIN bseg AS b ##ITAB_KEY_IN_SELECT
         ON a~rbukrs = b~bukrs AND a~gjahr  = b~gjahr AND a~belnr  = b~belnr AND a~buzei  = b~buzei
      FIELDS b~bukrs, b~belnr, b~gjahr, b~buzei, b~xnegp, b~shkzg
      INTO TABLE @DATA(lt_bsegnew).
    IF sy-subrc = 0. SORT lt_bsegnew BY bukrs belnr gjahr buzei. ENDIF.

    " =====================================================================
    " 4. 映射到展示内表
    " =====================================================================
    LOOP AT me->lt_find ASSIGNING <ls_find>.
      APPEND INITIAL LINE TO lt_alv_display ASSIGNING FIELD-SYMBOL(<ls_alv_display>).
      <ls_alv_display>-rbukrs = <ls_find>-rbukrs.
      <ls_alv_display>-lifnr  = <ls_find>-lifnr.

      READ TABLE lt_lfa1 INTO DATA(ls_lfa1) WITH KEY lifnr = <ls_find>-lifnr BINARY SEARCH.
      IF sy-subrc = 0.
        <ls_alv_display>-name1 = ls_lfa1-name. <ls_alv_display>-gork  = '供应商'.
      ELSE.
        READ TABLE lt_kna1 INTO DATA(ls_kna1) WITH KEY kunnr = <ls_find>-lifnr BINARY SEARCH.
        IF sy-subrc = 0.
          <ls_alv_display>-name1 = ls_kna1-name. <ls_alv_display>-gork  = '客户'.
        ENDIF.
      ENDIF.

      <ls_alv_display>-gjahr  = <ls_find>-gjahr.
      <ls_alv_display>-poper  = <ls_find>-poper.
      <ls_alv_display>-racct  = <ls_find>-racct.
      <ls_alv_display>-budat  = <ls_find>-budat.

      READ TABLE lt_skat INTO DATA(ls_skat) WITH KEY saknr = <ls_find>-racct BINARY SEARCH.
      IF sy-subrc = 0. <ls_alv_display>-txt50 = ls_skat-txt50. ENDIF.

      <ls_alv_display>-belnr  = <ls_find>-belnr.

      READ TABLE lt_zfit_ofn INTO DATA(ls_zfit_ofn) WITH KEY bukrs = <ls_find>-rbukrs gjahr = <ls_find>-gjahr belnr = <ls_find>-belnr BINARY SEARCH.
      IF sy-subrc = 0. <ls_alv_display>-hfc_doc = ls_zfit_ofn-hfc_doc. ENDIF.

      <ls_alv_display>-bktxt  = <ls_find>-bktxt.
      <ls_alv_display>-rwcur  = <ls_find>-rwcur.
      <ls_alv_display>-rhcur  = <ls_find>-rhcur.

      READ TABLE lt_bsegnew INTO DATA(ls_bsegnew) WITH KEY bukrs = <ls_find>-rbukrs gjahr = <ls_find>-gjahr belnr = <ls_find>-belnr buzei = <ls_find>-buzei BINARY SEARCH.
      IF sy-subrc = 0.
        IF ls_bsegnew-xnegp = 'X'.
          IF ls_bsegnew-shkzg = 'S'.
            <ls_alv_display>-dkfwsl = <ls_find>-wsl. <ls_alv_display>-dkfhsl = <ls_find>-hsl.
          ELSEIF ls_bsegnew-shkzg = 'H'.
            <ls_alv_display>-wsl    = <ls_find>-wsl. <ls_alv_display>-hsl    = <ls_find>-hsl.
          ENDIF.
        ELSE.
          IF <ls_find>-hsl >= 0.
            <ls_alv_display>-wsl    = <ls_find>-wsl. <ls_alv_display>-hsl    = <ls_find>-hsl.
          ELSE.
            <ls_alv_display>-dkfwsl = <ls_find>-wsl. <ls_alv_display>-dkfhsl = <ls_find>-hsl.
          ENDIF.
        ENDIF.
      ELSE.
        IF <ls_find>-hsl >= 0.
          <ls_alv_display>-wsl    = <ls_find>-wsl. <ls_alv_display>-hsl    = <ls_find>-hsl.
        ELSE.
          <ls_alv_display>-dkfwsl = <ls_find>-wsl. <ls_alv_display>-dkfhsl = <ls_find>-hsl.
        ENDIF.
      ENDIF.
      <ls_alv_display>-drcrkd = <ls_find>-drcrk.
    ENDLOOP.

    " =====================================================================
    " 5. 【严控排序与独立账本滚算】
    " =====================================================================
    SORT lt_alv_display BY rbukrs ASCENDING
                           lifnr  ASCENDING
                           racct  ASCENDING
                           rwcur  ASCENDING
                           rhcur  ASCENDING
                           gjahr  ASCENDING
                           poper  ASCENDING.

    TYPES: BEGIN OF ty_run_bal,
             key    TYPE string,
             beghsl TYPE fins_vhcur12,
             begwsl TYPE fins_vwcur12,
           END OF ty_run_bal.
    DATA: lt_run_bal   TYPE HASHED TABLE OF ty_run_bal WITH UNIQUE KEY key,
          lv_curr_key  TYPE string,
          ls_beg_match TYPE ty_beg_bal.

    LOOP AT lt_alv_display ASSIGNING FIELD-SYMBOL(<ls_calc>).
      lv_curr_key = |{ <ls_calc>-rbukrs }-{ <ls_calc>-lifnr }-{ <ls_calc>-racct }-{ <ls_calc>-rwcur }-{ <ls_calc>-rhcur }|.

      READ TABLE lt_run_bal ASSIGNING FIELD-SYMBOL(<ls_run_bal>) WITH TABLE KEY key = lv_curr_key.
      IF sy-subrc <> 0.
        DATA(ls_new_bal) = VALUE ty_run_bal( key = lv_curr_key ).
        READ TABLE lt_beg_bal INTO ls_beg_match
             WITH KEY rbukrs = <ls_calc>-rbukrs
                      lifnr  = <ls_calc>-lifnr
                      racct  = <ls_calc>-racct
                      rwcur  = <ls_calc>-rwcur
                      rhcur  = <ls_calc>-rhcur BINARY SEARCH.
        IF sy-subrc = 0.
          ls_new_bal-beghsl = ls_beg_match-beghsl.
          ls_new_bal-begwsl = ls_beg_match-begwsl.
        ENDIF.
        INSERT ls_new_bal INTO TABLE lt_run_bal ASSIGNING <ls_run_bal>.
      ENDIF.

      <ls_calc>-beghsl = <ls_run_bal>-beghsl.
      <ls_calc>-begwsl = <ls_run_bal>-begwsl.

      <ls_calc>-endhsl = <ls_calc>-beghsl + <ls_calc>-hsl - <ls_calc>-dkfhsl.
      <ls_calc>-endwsl = <ls_calc>-begwsl + <ls_calc>-wsl - <ls_calc>-dkfwsl.

      IF <ls_calc>-endhsl > 0.
        <ls_calc>-drcrkt = '借'.
      ELSEIF <ls_calc>-endhsl < 0.
        <ls_calc>-drcrkt = '贷'.
      ELSE.
        <ls_calc>-drcrkt = '平'.
      ENDIF.

      <ls_run_bal>-beghsl = <ls_calc>-endhsl.
      <ls_run_bal>-begwsl = <ls_calc>-endwsl.
    ENDLOOP.

    " =====================================================================
    " 6. 【终极修复】镜像追踪法：彻底告别计算偏差与串行
    " =====================================================================
    CLEAR lt_final_display.
    CLEAR: ls_sum_month, ls_sum_year.

    DATA: lv_is_new_month TYPE abap_bool VALUE abap_true,
          lv_is_new_year  TYPE abap_bool VALUE abap_true.

    DATA: lv_month_beghsl TYPE fins_vhcur12,
          lv_month_begwsl TYPE fins_vwcur12,
          lv_year_beghsl  TYPE fins_vhcur12,
          lv_year_begwsl  TYPE fins_vwcur12.

    LOOP AT lt_alv_display INTO DATA(ls_current).

      " 【核心修复点】在执行任何可能会重置系统变量（如INSERT）的操作之前，优先捕获当前行索引
      DATA(lv_current_index) = sy-tabix.

      " --- 动态组装包含年度与期间的分组键 (保证每月生成一个期初) ---
      DATA(lv_group_key) = |{ ls_current-rbukrs }-{ ls_current-lifnr }-{ ls_current-racct }-{ ls_current-rwcur }-{ ls_current-gjahr }-{ ls_current-poper }|.

      IF NOT line_exists( lt_processed_keys[ table_line = lv_group_key ] ).
        " 1. 构造期初行
        DATA: ls_opening TYPE ts_s_alvplay.
        ls_opening-bktxt = '期初余额'.
        ls_opening-endhsl = ls_current-beghsl.
        ls_opening-endwsl = ls_current-begwsl.

        " 设置期初余额的借贷方向
        IF ls_opening-endhsl > 0.
          ls_opening-drcrkt = '借'.
        ELSEIF ls_opening-endhsl < 0.
          ls_opening-drcrkt = '贷'.
        ELSE.
          ls_opening-drcrkt = '平'.
        ENDIF.

        " 2. 插入到最终输出表
        APPEND ls_opening TO lt_final_display.

        " 3. 标记该月度分组已成功生成，避免重复插入
        INSERT lv_group_key INTO TABLE lt_processed_keys.
      ENDIF.

      " --- 捕捉每一段新期间/年度的明细第一行金额作为期初追踪基准 ---
      IF lv_is_new_year = abap_true.
        lv_year_beghsl = ls_current-beghsl.
        lv_year_begwsl = ls_current-begwsl.
        lv_is_new_year = abap_false.
      ENDIF.

      IF lv_is_new_month = abap_true.
        lv_month_beghsl = ls_current-beghsl.
        lv_month_begwsl = ls_current-begwsl.
        lv_is_new_month = abap_false.
      ENDIF.

      " 插入明细数据行
      APPEND ls_current TO lt_final_display.

      " 发生额累加逻辑
      ls_sum_month-wsl    = ls_sum_month-wsl + ls_current-wsl.
      ls_sum_month-hsl    = ls_sum_month-hsl + ls_current-hsl.
      ls_sum_month-dkfwsl = ls_sum_month-dkfwsl + ls_current-dkfwsl.
      ls_sum_month-dkfhsl = ls_sum_month-dkfhsl + ls_current-dkfhsl.

      ls_sum_year-wsl     = ls_sum_year-wsl + ls_current-wsl.
      ls_sum_year-hsl     = ls_sum_year-hsl + ls_current-hsl.
      ls_sum_year-dkfwsl  = ls_sum_year-dkfwsl + ls_current-dkfwsl.
      ls_sum_year-dkfhsl  = ls_sum_year-dkfhsl + ls_current-dkfhsl.

      " 探查下一行状态 (使用受保护的局部索引变量，防范 LOOK AHEAD 机制失效)
      DATA: ls_next TYPE ts_s_alvplay.
      CLEAR ls_next.
      DATA(lv_next_idx) = lv_current_index + 1.
      READ TABLE lt_alv_display INTO ls_next INDEX lv_next_idx.
      DATA(lv_is_last_row) = sy-subrc.

      " -------------------------------------------------------------------
      " 【输出 1：本月合计】
      " -------------------------------------------------------------------
      IF lv_is_last_row <> 0 OR
         ls_next-rbukrs <> ls_current-rbukrs OR
         ls_next-lifnr  <> ls_current-lifnr  OR
         ls_next-racct  <> ls_current-racct  OR
         ls_next-rwcur  <> ls_current-rwcur  OR
         ls_next-rhcur  <> ls_current-rhcur  OR
         ls_next-gjahr  <> ls_current-gjahr  OR
         ls_next-poper  <> ls_current-poper.

        ls_sum_month-rbukrs = ls_current-rbukrs.
        ls_sum_month-lifnr  = ls_current-lifnr.
        ls_sum_month-name1  = ls_current-name1.
        ls_sum_month-gjahr  = ls_current-gjahr.
        ls_sum_month-poper  = ls_current-poper.
        ls_sum_month-racct  = ls_current-racct.
        ls_sum_month-rwcur  = ls_current-rwcur.
        ls_sum_month-rhcur  = ls_current-rhcur.
        ls_sum_month-bktxt  = '本月合计'.
        
        " 🟩 整行变绿逻辑：将 fname 设为空字符串，实现全行着色 🟩
        CLEAR: lt_s_color, ls_s_color.
        ls_s_color-fname     = ''.          " <--- 留空则代表整行变色
        ls_s_color-color-col = col_positive." 绿色 (5)
        ls_s_color-color-int = 1.           " 高亮
        APPEND ls_s_color TO lt_s_color.
        ls_sum_month-t_color = lt_s_color.

        ls_sum_month-beghsl = lv_month_beghsl.
        ls_sum_month-begwsl = lv_month_begwsl.
        ls_sum_month-endhsl = ls_current-endhsl.
        ls_sum_month-endwsl = ls_current-endwsl.

        IF ls_sum_month-endhsl > 0.
          ls_sum_month-drcrkt = '借'.
        ELSEIF ls_sum_month-endhsl < 0.
          ls_sum_month-drcrkt = '贷'.
        ELSE.
          ls_sum_month-drcrkt = '平'.
        ENDIF.

        APPEND ls_sum_month TO lt_final_display.
        CLEAR ls_sum_month.
        lv_is_new_month = abap_true. 
      ENDIF.

      " -------------------------------------------------------------------
      " 【输出 2：本年累计】
      " -------------------------------------------------------------------
      IF lv_is_last_row <> 0 OR
         ls_next-rbukrs <> ls_current-rbukrs OR
         ls_next-lifnr  <> ls_current-lifnr  OR
         ls_next-racct  <> ls_current-racct  OR
         ls_next-rwcur  <> ls_current-rwcur  OR
         ls_next-rhcur  <> ls_current-rhcur  OR
         ls_next-gjahr  <> ls_current-gjahr.

        ls_sum_year-rbukrs = ls_current-rbukrs.
        ls_sum_year-lifnr  = ls_current-lifnr.
        ls_sum_year-name1  = ls_current-name1.
        ls_sum_year-gjahr  = ls_current-gjahr.
        ls_sum_year-poper  = ls_current-poper.
        ls_sum_year-racct  = ls_current-racct.
        ls_sum_year-rwcur  = ls_current-rwcur.
        ls_sum_year-rhcur  = ls_current-rhcur.
        ls_sum_year-bktxt  = '本年累计'.
        
        " 🟥 整行变红逻辑：将 fname 设为空字符串，实现全行着色 🟥
        CLEAR: lt_s_color, ls_s_color.
        ls_s_color-fname     = ''.          " <--- 留空则代表整行变色
        ls_s_color-color-col = 6.           " 红色 (6)
        ls_s_color-color-int = 1.           " 高亮
        APPEND ls_s_color TO lt_s_color.
        ls_sum_year-t_color = lt_s_color.

        ls_sum_year-beghsl = lv_year_beghsl.
        ls_sum_year-begwsl = lv_year_begwsl.
        ls_sum_year-endhsl = ls_current-endhsl.
        ls_sum_year-endwsl = ls_current-endwsl.

        IF ls_sum_year-endhsl > 0.
          ls_sum_year-drcrkt = '借'.
        ELSEIF ls_sum_year-endhsl < 0.
          ls_sum_year-drcrkt = '贷'.
        ELSE.
          ls_sum_year-drcrkt = '平'.
        ENDIF.

        APPEND ls_sum_year TO lt_final_display.
        CLEAR ls_sum_year.
        lv_is_new_year = abap_true. 
      ENDIF.

    ENDLOOP.

    lt_alv_display = lt_final_display.

  ENDMETHOD.

```

---

### 2. 逻辑规范与问题修复详解 (English Logic Specification)

Here is the structured documentation explaining how the data flows and why the previous duplicate total issue happened, written for technical reference:

#### 1. Bug Cause Analysis (Why the 'Yearly Total' duplicated)

In ABAP, when executing an `INSERT ... INTO TABLE HASHED_TABLE` statement, the system variable `sy-tabix` is automatically reset to `0`.
In the previous logic, `DATA(lv_current_index) = sy-tabix` was placed *after* the Opening Balance injection. Because the hash table `INSERT` happened right before it, `sy-tabix` became `0`. Consequently, the "Look Ahead" mechanism calculated `lv_next_idx = 0 + 1 = 1`. The program was continuously inspecting the **first row** of the display table instead of the actual next record, triggering false boundary conditions and duplicating total blocks inside ledger segments.

* **Fix**: Captured `sy-tabix` into a local protected immutable variable `lv_current_index` at the **absolute entry point** of the loop before any data modification occurs.

#### 2. Monthly Rolling Opening Balance Insertion

To generate an independent "Opening Balance (期初余额)" block for every single ledger combination per fiscal month:

* **Granular Hash Key**: Enlarged the `lv_group_key` signature to encompass the fiscal year (`gjahr`) and posting period (`poper`).

$$\text{Key} = \text{Company} - \text{Customer/Vendor} - \text{G/L Account} - \text{Currency} - \mathbf{\text{Year}} - \mathbf{\text{Period}}$$


* **Conditional Trigger**: Utilizing `line_exists()`, the program evaluates if this unique chronological ledger key has been created. If not, it clones the core context, labels the narration text (`bktxt`) as `'期初余额'`, maps the beginning balances, and maps the debit/credit vectors before injecting it as a distinct row block.

#### 3. Full-Row Highlighting Optimization

By default, mapping a field catalog attribute to a specific structural field name narrows color rendering down to a single cell grid.

* **Mechanism**: To override cell-level painting and implement full-row formatting for total blocks, the technical field string reference inside the `LVC_S_SCOL` instance must be left unassigned:

$$\text{ls\_s\_color-fname} = \text{''}$$


* When the ALV control framework identifies an empty string within the layout matrix color table (`t_color`), it maps the specified SAP UI palette indexing code (`col_positive` / `6`) across the entire horizontal grid track.